# Cleanup: Delete Workshop Resources

This notebook deletes all resources created in **Module 6** (production deployment) and **Module 7** (long-term memory).

⚠️ **WARNING:** This will permanently delete:
- AgentCore Gateway and Runtime
- Lambda functions (8 total)
- Lambda Layer (neo4j driver)
- DynamoDB tables (Hotels, Bookings, SteeringRules)
- IAM roles
- ECR repositories
- CodeBuild projects
- Memory resources

**Cost after cleanup:** $0 (all resources deleted)

Run each cell in order to delete resources safely.

In [ ]:
import boto3
import time
import os

# Configuration (must match Module 6)
REGION = os.environ.get("AWS_REGION", "us-east-1")
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]

# Resource names
HOTELS_TABLE = "workshop-Hotels"
BOOKINGS_TABLE = "workshop-Bookings"
STEERING_RULES_TABLE = "workshop-SteeringRules"
LAMBDA_ROLE_NAME = "workshop-LambdaExecutionRole"
AGENTCORE_ROLE_NAME = "workshop-AgentCoreExecutionRole"
GATEWAY_NAME = "HotelBookingGateway"
RUNTIME_NAME = "HotelBookingAgent"
MEMORY_RUNTIME_NAME = "HotelBookingAgentWithMemory"
MEMORY_NAME = "workshop_HotelBookingMemory"

# Lambda function names
LAMBDA_FUNCTIONS = [
    "hotel-booking-search_available_hotels",
    "hotel-booking-book_hotel",
    "hotel-booking-get_booking",
    "hotel-booking-process_payment",
    "hotel-booking-confirm_booking",
    "hotel-booking-cancel_booking",
    "hotel-booking-validate_booking_rules",
    "hotel-booking-query_knowledge_graph",  # May not exist if Neo4j not available
]

# AWS clients
dynamodb = boto3.client("dynamodb", region_name=REGION)
iam = boto3.client("iam")
lambda_client = boto3.client("lambda", region_name=REGION)
agentcore = boto3.client("bedrock-agentcore-control", region_name=REGION)
ecr = boto3.client("ecr", region_name=REGION)
codebuild = boto3.client("codebuild", region_name=REGION)

print(f"Account:  {ACCOUNT_ID}")
print(f"Region:   {REGION}")
print("\n⚠️  Ready to delete resources. Run the cells below to proceed.")

---
## Step 1: Delete AgentCore Memory (Module 7)

If you ran Module 7, delete the Memory resource first before deleting the memory-enabled agent runtime.

In [ ]:
# Delete Memory resource
try:
    memories = agentcore.list_memories()["memories"]
    memory = next((m for m in memories if m["memoryName"] == MEMORY_NAME), None)
    if memory:
        memory_id = memory["memoryId"]
        agentcore.delete_memory(memoryId=memory_id)
        print(f"✅ Deleted Memory: {memory_id}")
    else:
        print("ℹ️  Memory not found (may not have been created)")
except Exception as e:
    print(f"⚠️  Memory deletion failed: {e}")

---
## Step 2: Delete AgentCore Runtimes

Delete both agent runtimes:
1. `HotelBookingAgent` (Module 6)
2. `HotelBookingAgentWithMemory` (Module 7, if created)

In [ ]:
# Delete both agent runtimes
for runtime_name in [RUNTIME_NAME, MEMORY_RUNTIME_NAME]:
    try:
        runtimes = agentcore.list_agent_runtimes()["agentRuntimes"]
        rt = next((r for r in runtimes if r["agentRuntimeName"] == runtime_name), None)
        if rt:
            runtime_id = rt["agentRuntimeId"]
            agentcore.delete_agent_runtime(agentRuntimeId=runtime_id)
            print(f"✅ Deleting runtime: {runtime_name} ({runtime_id})...")
            
            # Wait for deletion
            for _ in range(30):
                try:
                    agentcore.get_agent_runtime(agentRuntimeId=runtime_id)
                    time.sleep(5)
                except:
                    print(f"   ✅ Runtime deleted: {runtime_name}")
                    break
        else:
            print(f"ℹ️  Runtime not found: {runtime_name}")
    except Exception as e:
        print(f"⚠️  Runtime deletion failed ({runtime_name}): {e}")

---
## Step 3: Delete AgentCore Gateway

Delete the Gateway and all its targets (Lambda tool registrations).

In [ ]:
# Find and delete Gateway
try:
    gateways = agentcore.list_gateways()["items"]
    gw = next((g for g in gateways if g["name"] == GATEWAY_NAME), None)
    if gw:
        gateway_id = gw["gatewayId"]
        
        # Delete all targets first
        try:
            targets = agentcore.list_gateway_targets(gatewayId=gateway_id)["items"]
            for target in targets:
                try:
                    agentcore.delete_gateway_target(
                        gatewayId=gateway_id,
                        gatewayTargetIdentifier=target["gatewayTargetId"]
                    )
                    print(f"   ✅ Deleted target: {target['name']}")
                except Exception as e:
                    print(f"   ⚠️  Target deletion failed: {target['name']}: {e}")
        except Exception:
            pass
        
        # Delete Gateway
        agentcore.delete_gateway(gatewayId=gateway_id)
        print(f"✅ Deleted Gateway: {gateway_id}")
    else:
        print("ℹ️  Gateway not found")
except Exception as e:
    print(f"⚠️  Gateway deletion failed: {e}")

---
## Step 4: Delete Lambda Functions

Delete all 8 Lambda functions (7 booking tools + 1 Neo4j query).

In [ ]:
# Delete all Lambda functions
for func_name in LAMBDA_FUNCTIONS:
    try:
        lambda_client.delete_function(FunctionName=func_name)
        print(f"✅ Deleted Lambda: {func_name}")
    except lambda_client.exceptions.ResourceNotFoundException:
        print(f"ℹ️  Lambda not found: {func_name}")
    except Exception as e:
        print(f"⚠️  Lambda deletion failed ({func_name}): {e}")

---
## Step 5: Delete Lambda Layer

Delete the Neo4j driver Lambda Layer (all versions).

In [ ]:
# Delete Lambda Layer (all versions)
try:
    layer_name = "workshop-neo4j-driver"
    versions = lambda_client.list_layer_versions(LayerName=layer_name)["LayerVersions"]
    for v in versions:
        lambda_client.delete_layer_version(LayerName=layer_name, VersionNumber=v["Version"])
        print(f"✅ Deleted layer version: {layer_name}:{v['Version']}")
except lambda_client.exceptions.ResourceNotFoundException:
    print("ℹ️  Lambda Layer not found")
except Exception as e:
    print(f"⚠️  Layer deletion failed: {e}")

---
## Step 6: Delete DynamoDB Tables

Delete the three DynamoDB tables: Hotels, Bookings, SteeringRules.

In [ ]:
# Delete DynamoDB tables
for table_name in [HOTELS_TABLE, BOOKINGS_TABLE, STEERING_RULES_TABLE]:
    try:
        dynamodb.delete_table(TableName=table_name)
        print(f"✅ Deleted DynamoDB table: {table_name}")
    except dynamodb.exceptions.ResourceNotFoundException:
        print(f"ℹ️  Table not found: {table_name}")
    except Exception as e:
        print(f"⚠️  Table deletion failed ({table_name}): {e}")

---
## Step 7: Delete IAM Roles

Delete the two IAM roles:
1. `workshop-LambdaExecutionRole`
2. `workshop-AgentCoreExecutionRole`

All inline policies and attached managed policies are detached first.

In [ ]:
# Delete IAM roles
for role_name in [LAMBDA_ROLE_NAME, AGENTCORE_ROLE_NAME]:
    try:
        # Detach managed policies
        attached = iam.list_attached_role_policies(RoleName=role_name)["AttachedPolicies"]
        for p in attached:
            iam.detach_role_policy(RoleName=role_name, PolicyArn=p["PolicyArn"])
            print(f"   Detached managed policy: {p['PolicyName']}")
        
        # Delete inline policies
        inline = iam.list_role_policies(RoleName=role_name)["PolicyNames"]
        for p in inline:
            iam.delete_role_policy(RoleName=role_name, PolicyName=p)
            print(f"   Deleted inline policy: {p}")
        
        # Delete role
        iam.delete_role(RoleName=role_name)
        print(f"✅ Deleted IAM role: {role_name}")
    except iam.exceptions.NoSuchEntityException:
        print(f"ℹ️  Role not found: {role_name}")
    except Exception as e:
        print(f"⚠️  Role deletion failed ({role_name}): {e}")

---
## Step 8: Delete ECR Repositories

Delete ECR repositories created by the bedrock-agentcore-starter-toolkit.

Repository names follow the pattern: `bedrock-agentcore-<runtime-name-lowercase>`

In [ ]:
# Delete ECR repositories
for runtime_name in [RUNTIME_NAME, MEMORY_RUNTIME_NAME]:
    repo_name = f"bedrock-agentcore-{runtime_name.lower()}"
    try:
        ecr.delete_repository(repositoryName=repo_name, force=True)
        print(f"✅ Deleted ECR repo: {repo_name}")
    except ecr.exceptions.RepositoryNotFoundException:
        print(f"ℹ️  ECR repo not found: {repo_name}")
    except Exception as e:
        print(f"⚠️  ECR deletion failed ({repo_name}): {e}")

---
## Step 9: Delete CodeBuild Projects

Delete CodeBuild projects created by the bedrock-agentcore-starter-toolkit.

Project names follow the pattern: `bedrock-agentcore-<runtime-name-lowercase>-builder`

In [ ]:
# Delete CodeBuild projects
for runtime_name in [RUNTIME_NAME, MEMORY_RUNTIME_NAME]:
    project_name = f"bedrock-agentcore-{runtime_name.lower()}-builder"
    try:
        codebuild.delete_project(name=project_name)
        print(f"✅ Deleted CodeBuild project: {project_name}")
    except codebuild.exceptions.ResourceNotFoundException:
        print(f"ℹ️  CodeBuild project not found: {project_name}")
    except Exception as e:
        print(f"⚠️  CodeBuild deletion failed ({project_name}): {e}")

---
## Step 10: Delete Local Config Files

Delete bedrock-agentcore-starter-toolkit config files from the local directory.

In [ ]:
import glob

# Delete local config files
for pattern in [".bedrock_agentcore*.yaml", os.path.expanduser("~/.bedrock_agentcore*.yaml")]:
    for cfg in glob.glob(pattern):
        try:
            os.remove(cfg)
            print(f"✅ Deleted config: {cfg}")
        except Exception as e:
            print(f"⚠️  Config deletion failed ({cfg}): {e}")

---
## Cleanup Complete

✅ All workshop resources have been deleted.

**Resources removed:**
- AgentCore Gateway and 2 Runtimes
- 8 Lambda functions + 1 Lambda Layer
- 3 DynamoDB tables
- 2 IAM roles
- 2 ECR repositories
- 2 CodeBuild projects
- 1 Memory resource

**What was NOT deleted:**
- Neo4j infrastructure (EC2 Code Editor with local Neo4j, or Central Neo4j ECS stack)
  - These are created by CloudFormation in Module 1, not by Module 6/7
  - Delete via AWS Console → CloudFormation → Delete Stack

**Cost:** $0 after cleanup (no ongoing charges)